# Data Preparation

## Objective

Prepare a candidate feature set for the enrollment-time student-outcome prediction task.

This notebook will:

1. Load the original dataset.
2. identify and exclude first- and second-semester features.
3. Separate the candidate input features from the target.
4. Verify the resulting dimensions and target distribution.

This is a preliminary preparation stage. Features whose availability at enrollment is uncertain will remain subject to review before the final model is trained.

The raw dataset will not be modified.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "data.csv"

df = pd.read_csv(data_path, sep=";")
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Target column exists:", "Target" in df.columns)

Dataset shape: (4424, 37)
Target column exists: True


## Excluding Post-Enrollment Features

The intended prediction point is enrollment. The first- and second-semester features describe academic activity or performance that occurs after that point.

These features are excluded from the candidate enrollment-time input set to avoid temporal data leakage.

In [2]:
semester_columns = [
    col for col in df.columns
    if "1st sem" in col.lower()
    or "2nd sem" in col.lower()
]

print("Number of semester-related columns:", len(semester_columns))

for col in semester_columns:
    print("-", col)

Number of semester-related columns: 12
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 1st sem (approved)
- Curricular units 1st sem (grade)
- Curricular units 1st sem (without evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)


## Candidate Features and Target

The target `Target` contains the outcome the model will attempt to predict.

The candidate inputs consist of all remaining columns after removing the target and the 12 semester-related features.

These are candidate inputs, not yet the final approved feature set. Variables whose timing is uncertain will be reviewed before modeling.

In [3]:
candidate_feature_columns = [
    col for col in df.columns
    if col != "Target" and col not in semester_columns
]

X_candidate = df[candidate_feature_columns].copy()
y = df["Target"].copy()

print("Candidate feature count:", len(candidate_feature_columns))
print("Candidate input shape:", X_candidate.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Candidate feature count: 24
Candidate input shape: (4424, 24)
Target shape: (4424,)

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [4]:
assert "Target" not in X_candidate.columns

assert not any(
    "1st sem" in col.lower() or "2nd sem" in col.lower()
    for col in X_candidate.columns
)

assert len(X_candidate) == len(y)

print("All checks passed.")
print("No semester-related columns remain in X_candidate.")
print("Every candidate input row has a corresponding target.")

All checks passed.
No semester-related columns remain in X_candidate.
Every candidate input row has a corresponding target.
